# consumer-agents — Blog Figures

Produces three publication-quality figures and one text comparison for the blog post.

Output: PNGs saved to `notebooks/figures/` and a markdown snippet printed at the end.

Figures:
- **V1** — daily purchase fingerprint grid (3 personas × 3 runs)
- **V2** — concentration drop bar chart
- **V4** — settling-vocab over time (3 personas, 2 runs)
- **V3** — side-by-side Maya week-6 reflections, as a text block ready to paste


## Setup


In [ ]:
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

from consumer_agents.datalake.queries import open_run
from consumer_agents.world.catalog import load_catalog

# Runs (edit if you regenerate with different runs)
RUN_PATHS = {
    "baseline":     Path("../runs/baseline-d862c721"),
    "no_refl":      Path("../runs/baseline_no_reflections-72d5b0bb"),
    "first_person": Path("../runs/baseline_first_person-8ec49f4d"),
}
CONS = {label: open_run(p) for label, p in RUN_PATHS.items()}

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

# Load the catalog so we can map sku_id -> human-readable name in legends/labels.
_catalog = load_catalog("../world")
SKU_NAME = {sku.id: sku.name for sku in _catalog.skus}

def sku_label(sku_id):
    """Pretty name for a SKU id; falls back to the id if not found."""
    return SKU_NAME.get(sku_id, sku_id)

SHORT = {
    "persona-maya-001":  "Maya",
    "persona-raj-001":   "Raj",
    "persona-elena-001": "Elena",
}
AGENTS = ["persona-maya-001", "persona-raj-001", "persona-elena-001"]
RUN_LABELS_PRETTY = {
    "baseline":     "baseline (third-person)",
    "no_refl":      "no reflections",
    "first_person": "first-person diary",
}

# Shared style — keep it minimal, blog-friendly
plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":         11,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.titlesize":    12,
    "axes.titleweight":  "semibold",
    "axes.labelsize":    10,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "legend.fontsize":   9,
    "figure.dpi":        150,
    "savefig.dpi":       150,
    "savefig.bbox":      "tight",
    "savefig.facecolor": "white",
})

print(f"Loaded {len(CONS)} runs and {len(SKU_NAME)} SKU names. "
      f"Figures will save to {FIG_DIR.resolve()}")


## V1 — Daily purchase fingerprint grid

3 personas × 3 runs = 9 strips. Each strip is 90 days wide. Each cell is colored by the dominant SKU bought that day. White = no purchase. The baseline column should be visibly monochrome (mostly coffee). The first-person column should be visibly varied.

Legend uses the catalog's human-readable SKU names (e.g., *Coffee Shop Drip Coffee* instead of `sku-dine-001`).


In [ ]:
def daily_dominant_sku(con, agent_id, n_days=90):
    """For each day, return the most-cart_added SKU (or None). Day offset is computed in DuckDB."""
    df = con.execute(f"""
      WITH start AS (SELECT MIN(tick_day) AS d0 FROM snapshots WHERE agent_id='{agent_id}')
      SELECT (tick_day - (SELECT d0 FROM start)) AS day_offset,
             sku_id, COUNT(*) AS n
      FROM v_cart_add WHERE agent_id='{agent_id}'
      GROUP BY 1, 2
    """).fetchdf()
    out = [None] * n_days
    if df.empty:
        return out
    dominant = (
        df.sort_values(["day_offset", "n"], ascending=[True, False])
          .groupby("day_offset").first().reset_index()[["day_offset", "sku_id"]]
    )
    for _, row in dominant.iterrows():
        d = int(row["day_offset"])
        if 0 <= d < n_days:
            out[d] = row["sku_id"]
    return out

# Build color map: top SKUs across all runs get distinctive colors, rest gray
all_sku_counts = (
    pd.concat([
        con.execute("SELECT sku_id, COUNT(*) AS n FROM v_cart_add GROUP BY 1").fetchdf()
        for con in CONS.values()
    ]).groupby("sku_id")["n"].sum().sort_values(ascending=False)
)

# sku-dine-001 (the coffee) gets a memorable distinct color so the convergence pops
COFFEE = "sku-dine-001"
top_skus = [COFFEE] + [s for s in all_sku_counts.head(11).index if s != COFFEE][:10]

# Color palette — coffee is warm orange/yellow; others are tab10 spread
tab = plt.get_cmap("tab20")
color_map = {COFFEE: "#e89c3e"}  # coffee-shop orange
for i, sku in enumerate([s for s in top_skus if s != COFFEE]):
    color_map[sku] = tab(i % 20)
DEFAULT_COLOR = "#cccccc"  # gray for other SKUs
NONE_COLOR = "#f7f7f7"     # near-white for days with no purchase

def sku_color(sku):
    if sku is None:
        return NONE_COLOR
    return color_map.get(sku, DEFAULT_COLOR)

# Plot: 3 rows (personas) x 3 cols (runs), each cell is a strip of 90 day-cells
fig, axes = plt.subplots(3, 3, figsize=(11, 4.5), sharex=True, sharey=True)

for col, (run_label, con) in enumerate(CONS.items()):
    for row, agent in enumerate(AGENTS):
        ax = axes[row, col]
        skus = daily_dominant_sku(con, agent)
        for d, sku in enumerate(skus):
            ax.add_patch(plt.Rectangle((d, 0), 1, 1, color=sku_color(sku), linewidth=0))
        ax.set_xlim(0, 90)
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        ax.set_xticks([0, 30, 60, 90])
        if row == 0:
            ax.set_title(RUN_LABELS_PRETTY[run_label])
        if col == 0:
            ax.set_ylabel(SHORT[agent], rotation=0, ha="right", va="center", labelpad=10)
        if row == 2:
            ax.set_xlabel("day")
        ax.spines["left"].set_visible(False)
        ax.spines["bottom"].set_visible(False)
        ax.tick_params(left=False, bottom=False)

# Legend — use human-readable SKU names
legend_skus = [COFFEE] + [s for s in top_skus if s != COFFEE][:6]
patches = [mpatches.Patch(color=sku_color(s), label=sku_label(s)) for s in legend_skus]
patches.append(mpatches.Patch(color=DEFAULT_COLOR, label="other SKUs"))
patches.append(mpatches.Patch(color=NONE_COLOR, label="no purchase"))
fig.legend(handles=patches, loc="lower center", ncol=3,
           bbox_to_anchor=(0.5, -0.10), frameon=False)

fig.suptitle("Daily purchase fingerprint: 3 personas × 3 runs",
             fontsize=13, fontweight="semibold", y=1.02)
fig.text(0.5, 0.94,
         "Each strip = 90 days. Each cell = the dominant SKU bought that day.",
         ha="center", fontsize=9, color="#666")

plt.tight_layout()
out_path = FIG_DIR / "v1_daily_fingerprint.png"
plt.savefig(out_path, bbox_inches="tight", facecolor="white")
print(f"Saved {out_path}")
plt.show()


## V2 — Concentration drop bar chart

Top-SKU concentration % per persona per run. The headline number — every persona's baseline towers over the alternative runs.


In [ ]:
def concentration_pct(con, agent_id):
    row = con.execute(f"""
      WITH per_sku AS (
        SELECT sku_id, COUNT(*) AS n FROM v_cart_add
        WHERE agent_id='{agent_id}' GROUP BY 1
      )
      SELECT 100.0 * MAX(n) / SUM(n) AS pct FROM per_sku
    """).fetchone()
    return float(row[0]) if row and row[0] is not None else 0.0

data = {agent: {} for agent in AGENTS}
for run_label, con in CONS.items():
    for agent in AGENTS:
        data[agent][run_label] = concentration_pct(con, agent)

fig, ax = plt.subplots(figsize=(8, 4.5))

x = np.arange(len(AGENTS))
width = 0.27
run_order = ["baseline", "no_refl", "first_person"]
colors = {
    "baseline":     "#2b3b54",  # dark steel blue
    "no_refl":      "#a8b3c2",  # muted gray-blue
    "first_person": "#e89c3e",  # warm orange (intervention that broke through)
}

for i, run in enumerate(run_order):
    vals = [data[agent][run] for agent in AGENTS]
    offset = (i - 1) * width
    bars = ax.bar(x + offset, vals, width,
                  label=RUN_LABELS_PRETTY[run],
                  color=colors[run], edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 1.5,
                f"{v:.0f}%", ha="center", va="bottom", fontsize=9, color="#333")

ax.set_xticks(x)
ax.set_xticklabels([SHORT[a] for a in AGENTS])
ax.set_ylim(0, 100)
ax.set_ylabel("top-SKU concentration (% of cart_adds)")
ax.legend(frameon=False, loc="upper right")
ax.set_title("How much of each persona's cart was a single SKU?",
             fontsize=13, fontweight="semibold")
ax.grid(axis="y", linestyle="--", alpha=0.3)

fig.text(0.02, -0.03,
         "In the baseline, the most-bought SKU accounts for 69–83% of every persona's cart_adds.\n"
         "Removing reflections drops this to 33–43%. First-person diary reflections drop it further still.",
         fontsize=9, color="#666")

plt.tight_layout()
out_path = FIG_DIR / "v2_concentration.png"
plt.savefig(out_path, bbox_inches="tight", facecolor="white")
print(f"Saved {out_path}")
plt.show()


## V4 — Settling-vocab over time

Hits per reflection for words like *habit, infrastructure, automaticity, entrenched, routine, fixed, calcified, metabolic*. Tracks when the third-person model started narrating the basin. First-person reflections show this language at ~1/3 the rate.

(`no_refl` has no reflections, so it's omitted from this plot.)


In [ ]:
SETTLING_WORDS = [
    "habit", "habits", "habitual",
    "infrastructure", "automaticity", "automated", "automatic",
    "entrenched", "entrench",
    "routine", "routines",
    "fixed", "calcified", "metabolic",
    "streak", "loop",
]

def reflection_settling_series(con, agent_id):
    rows = con.execute(f"""
      SELECT tick_day, payload FROM events
      WHERE event_type='reflection' AND agent_id='{agent_id}'
      ORDER BY tick_day
    """).fetchall()
    if not rows:
        return pd.DataFrame(columns=["week_idx", "hits"])
    out = []
    for i, (td, payload) in enumerate(rows, start=1):
        text = " ".join(json.loads(payload).get("observations", [])).lower()
        hits = sum(len(re.findall(rf"\b{w}\b", text)) for w in SETTLING_WORDS)
        out.append({"week_idx": i, "hits": hits})
    return pd.DataFrame(out)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharey=True)

colors_line = {
    "baseline":     "#2b3b54",
    "first_person": "#e89c3e",
}

for ax, agent in zip(axes, AGENTS):
    for run_label in ["baseline", "first_person"]:
        if run_label not in CONS:
            continue
        df = reflection_settling_series(CONS[run_label], agent)
        if df.empty:
            continue
        ax.plot(df["week_idx"], df["hits"], marker="o",
                color=colors_line[run_label],
                label=RUN_LABELS_PRETTY[run_label], linewidth=2)
    ax.set_title(SHORT[agent])
    ax.set_xlabel("reflection week")
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.set_xticks([1, 4, 8, 12])

axes[0].set_ylabel("settling-vocab hits per reflection")
axes[-1].legend(frameon=False, loc="upper right", fontsize=9)

fig.suptitle("Crystallizing language in reflections, week-by-week",
             fontsize=13, fontweight="semibold", y=1.04)
fig.text(0.5, 0.96,
         '"habit / infrastructure / automaticity / entrenched / routine / calcified / metabolic / streak / loop" — count per reflection',
         ha="center", fontsize=9, color="#666")

plt.tight_layout()
out_path = FIG_DIR / "v4_settling_vocab.png"
plt.savefig(out_path, bbox_inches="tight", facecolor="white")
print(f"Saved {out_path}")
plt.show()


## V3 — Reflection quote comparison

Not a chart — a text comparison ready to paste into the blog. Pulls Maya's week-6 reflection from baseline and first-person runs.

Run this cell and copy the markdown output between the `---` lines into your blog post.


In [ ]:
def maya_week_reflection(con, week_idx=6):
    rows = con.execute("""
      SELECT tick_day, payload FROM events
      WHERE event_type='reflection' AND agent_id='persona-maya-001'
      ORDER BY tick_day
    """).fetchall()
    if len(rows) < week_idx:
        return None, []
    td, payload = rows[week_idx - 1]
    return td, json.loads(payload).get("observations", [])

td_base, obs_base = maya_week_reflection(CONS["baseline"])
td_fp, obs_fp = maya_week_reflection(CONS["first_person"])

print("---")
print()
print("### Maya, week 6 — same persona, same week, same data, different voice")
print()
print("**Baseline (third-person reflections):**")
print()
for o in obs_base:
    print(f"> {o}")
    print(">")
print()
print("**First-person diary reflections:**")
print()
for o in obs_fp:
    print(f"> {o}")
    print(">")
print()
print("---")


## Summary of saved figures


In [ ]:
for p in sorted(FIG_DIR.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:30}  {size_kb:>7.1f} KB")
